In [76]:
import pandas as pd
import yaml
import subprocess

In [77]:
REGENERATE_RULES = False 
REGENERATE_INDEX_RULES = False
INDEX_YML_PATH = "../data/diachronica/index_diachronica.yml"
RULES_YML_PATH = "data/asca_rules.yml"
RULES_CSV_PATH = "data/asca_rules.csv"
INDEX_RULES_CSV_PATH = "data/index_rules.csv"
DF_COLS = ["section_index", "section_name", "rule_id", "step_index", "c_index", "input", "output", "env", "exception"]

def split_condensed_rule(rule: dict) -> list[dict]:
    if "input" not in rule or "output" not in rule:
        raise ValueError("Input or output not found in rule")
    inputs = rule["input"].strip().split(" ")
    outputs = rule["output"].strip().split(" ")
    if (len(inputs) == len(outputs)) and len(inputs) > 1:
        return [{**rule, "input": i, "output": o} for i, o in zip(inputs, outputs)]
    return [rule]

if REGENERATE_RULES:
    cmd = ["uv", "run", "compile_rules", "--compiler-config", "../config/pipeline/compiler_config.yml", "--compiled-out", RULES_YML_PATH]

    output = subprocess.run(
        cmd, capture_output=True, timeout=60, text=True, check=True,
    )
    output.check_returncode()

    with open(RULES_YML_PATH, "r") as f:
        rules_doc = yaml.load(f, Loader=yaml.SafeLoader)

    rules = []
    for s in rules_doc["sections"]:
        for r in s["rules"]:
            if r.get("status", "") == "skipped" or "compile_error" in r:
                continue
            if "steps" not in r:
                for j, rule in enumerate(split_condensed_rule(r)):
                    rules.append({"section_index": s["index"], "section_name": s["section"], "step_index": 0, "c_index": j, **rule})
            else:
                for i,step in enumerate(r["steps"]):
                    for j, rule in enumerate(split_condensed_rule(step)):
                        rules.append({"section_index": s["index"], "section_name": s["section"], "step_index": i, "c_index": j, "rule_id": r["rule_id"], **rule})

    rules_df = pd.DataFrame(rules)[DF_COLS]
    rules_df.to_csv(RULES_CSV_PATH, index=False)
else:
    rules_df = pd.read_csv(RULES_CSV_PATH)

In [78]:
print(rules_df.shape)
rules_df.head(10)

(11994, 9)


,section_index,section_name,rule_id,step_index,c_index,input,output,env,exception
0,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dz-ʃ-tʃ,0,0,dz,ʒ,NaN,NaN
1,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dz-ʃ-tʃ,0,1,ʃ,F,NaN,NaN
2,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dz-ʃ-tʃ,0,2,tʃ,F,NaN,NaN
3,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dʒ,0,0,dʒ,tʃ,NaN,NaN
4,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dʒ,1,0,tʃ,ʃ,NaN,NaN
5,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-t,0,0,t,∅,_s#,NaN
6,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-ɬ,0,0,ɬ,l,NaN,NaN
7,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-f,0,0,f,p,NaN,NaN
8,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-aː,0,0,"{a,a:[+long]}","{e,e:[+long]}","_{ʕ,q}$",NaN
9,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-q-ʕ,0,0,q,ʔ,NaN,NaN


In [79]:
def extract_usable_rules(df: pd.DataFrame) -> pd.DataFrame:
    segments = pd.concat([df["input"], df["output"]])
    contexts = pd.concat([df["env"], df["exception"]]).dropna()

    min_segment_counts = 5
    min_context_counts = 5

    segments_counts = segments.value_counts()
    contexts_counts = contexts.value_counts()

    sample_segments = set(segments_counts[(segments_counts >= min_segment_counts)].index)
    sample_contexts = set(contexts_counts[(contexts_counts >= min_context_counts)].index)

    usable_inputs = df["input"].isin(sample_segments)
    usable_outputs = df["output"].isin(sample_segments)
    usable_envs = df["env"].isin(sample_contexts) | df["env"].isna()
    usable_exceptions = df["exception"].isin(sample_contexts) | df["exception"].isna()

    return df[usable_inputs & usable_outputs & usable_envs & usable_exceptions]

usable_rules_df = extract_usable_rules(rules_df)
print(usable_rules_df.shape)
usable_rules_df.head(10)


(6724, 9)


,section_index,section_name,rule_id,step_index,c_index,input,output,env,exception
0,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dz-ʃ-tʃ,0,0,dz,ʒ,NaN,NaN
1,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dz-ʃ-tʃ,0,1,ʃ,F,NaN,NaN
2,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dz-ʃ-tʃ,0,2,tʃ,F,NaN,NaN
3,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dʒ,0,0,dʒ,tʃ,NaN,NaN
4,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dʒ,1,0,tʃ,ʃ,NaN,NaN
5,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-t,0,0,t,∅,_s#,NaN
6,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-ɬ,0,0,ɬ,l,NaN,NaN
7,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-f,0,0,f,p,NaN,NaN
9,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-q-ʕ,0,0,q,ʔ,NaN,NaN
10,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-q-ʕ,0,1,ʕ,h,NaN,NaN


In [80]:
# try using cleaned index rules (they may be a bit simpler syntactically)

# just use the last validation run for now
COMPILED_RULES_PATH = "../diagnostics/inventory/rule-inventory-success.csv"
compiled_rules_df = pd.read_csv(COMPILED_RULES_PATH)[["section_index", "rule_id"]]
valid_rule_ids = set(compiled_rules_df["rule_id"])

if REGENERATE_INDEX_RULES:
    with open(INDEX_YML_PATH, "r") as f:
        rules_doc = yaml.load(f, Loader=yaml.SafeLoader)

    rules = []
    for s in rules_doc["sections"]:
        for r in s.get("rules", []):
            if r.get("status", "") == "skipped" or r["rule_id"] not in valid_rule_ids:
                continue
            if "stages" in r and len(r["stages"]) > 1:
                for i in range(len(r["stages"]) - 1):
                    rule = {
                        "input": r["stages"][i],
                        "output": r["stages"][i+1],
                        "rule_id": r["rule_id"],
                    }
                    if "env" in r:
                        rule["env"] = r["env"]
                    if "exception" in r:
                        rule["exception"] = r["exception"]
                        
                    for j, rl in enumerate(split_condensed_rule(rule)):
                        rules.append({"section_index": s["index"], "section_name": s["section"], "step_index": i, "c_index": j, **rl})

    index_rules_df = pd.DataFrame(rules)[DF_COLS]
    index_rules_df.to_csv(INDEX_RULES_CSV_PATH, index=False)
else:
    index_rules_df = pd.read_csv(INDEX_RULES_CSV_PATH)

print(index_rules_df.shape)
index_rules_df.head(10)

(11990, 9)


,section_index,section_name,rule_id,step_index,c_index,input,output,env,exception
0,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dz-ʃ-tʃ,0,0,dz,ʒ,NaN,NaN
1,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dz-ʃ-tʃ,0,1,ʃ,s₁,NaN,NaN
2,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dz-ʃ-tʃ,0,2,tʃ,s₂,NaN,NaN
3,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dʒ,0,0,dʒ,tʃ,NaN,NaN
4,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dʒ,1,0,tʃ,ʃ,NaN,NaN
5,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-t,0,0,t,∅,_s#,NaN
6,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-ɬ,0,0,ɬ,l,NaN,NaN
7,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-f,0,0,f,p,NaN,NaN
8,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-aː,0,0,a(ː),e(ː),"_{ʕ,q}$",NaN
9,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-q-ʕ,0,0,q,ʔ,NaN,NaN


In [ ]:
usable_index_rules_df = extract_usable_rules(index_rules_df)
print(usable_index_rules_df.shape)
usable_index_rules_df.head(10)

# TODO: fix the complex env/exception types (position, etc.)
# fails when REGENERATE_INDEX_RULES is True

(6654, 9)


,section_index,section_name,rule_id,step_index,c_index,input,output,env,exception
0,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dz-ʃ-tʃ,0,0,dz,ʒ,NaN,NaN
1,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dz-ʃ-tʃ,0,1,ʃ,s₁,NaN,NaN
2,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dz-ʃ-tʃ,0,2,tʃ,s₂,NaN,NaN
3,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dʒ,0,0,dʒ,tʃ,NaN,NaN
4,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dʒ,1,0,tʃ,ʃ,NaN,NaN
5,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-t,0,0,t,∅,_s#,NaN
6,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-ɬ,0,0,ɬ,l,NaN,NaN
7,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-f,0,0,f,p,NaN,NaN
9,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-q-ʕ,0,0,q,ʔ,NaN,NaN
10,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-q-ʕ,0,1,ʕ,h,NaN,NaN
